# instr_student: the student sees the query instruction too

So far only the TEACHER side used instructions (the 8K-embed teacher encoded the mix's 20,000 question texts with its card's query instruction); the student always saw plain text and was evaluated plain. Here the student's input of those same questions carries the same instruction (`student.query_prefix` in `configs/two_instr_student.yaml`), and the model is evaluated with it as the TR-MTEB query prompt. The lab trained `modernbert-tr-1k-embed` this way and gained 1.5 points from the prompt (61.41 -> 62.91, retrieval 53.6 -> 57.1).

Run (1k-base student, BGE-M3 + instructed 8K-embed, 200K mix, lr 1e-4, 5400 steps, seed 42; no new encoding, the instructed-teacher caches are reused):
1. `instr_student_s5400`

Reference: the same recipe with a plain student, `scale_two_instr8k_lr1e4` (TR-MTEB 67.40 plain). Evaluate three presets in an eval notebook (printed at the end): the new model plain, the new model with the prompt (only the 11 retrieval tasks run, the rest is copied from its plain run), and the OLD plain-trained student with the prompt (control: does the prompt help a model that never saw it?). Only differences above 0.3 are read. If the prompted score is clearly higher, the 10,800-step run (commented out in `RUNS`) is next. Re-running after a disconnect resumes; the last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# (name, config, extra overrides). The config already sets the 200K mix, lr 1e-4, 5400 steps and the student prefix.
CONFIG = 'configs/two_instr_student.yaml'
QUESTIONS = f'{DRIVE_ROOT}/data/mix/mix_v2_200k_questions.jsonl'
assert os.path.exists(QUESTIONS), QUESTIONS
RUNS = [
    ('instr_student_s5400', CONFIG, []),
    # next, only if the prompted score is clearly above 67.40:
    # ('instr_student_s10800', CONFIG, ['train.max_steps=10800']),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

# The prompt must be the training prefix exactly (real line break, no space after "Query:").
print('\nEval presets (paste into an eval notebook; plain first, then the prompted variants):')
print("STUDENT_INSTR = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\\nQuery:'")
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        path = f"f'{{DRIVE_ROOT}}/runs/{name}/{os.path.basename(os.path.dirname(d))}/final'"
        print(f"    'd_{name}': dict(model={path}),")
        print(f"    'd_{name}_instr': dict(model={path}, query_prompt=STUDENT_INSTR, reuse_from='d_{name}'),")
print("    'd_two_instr8k_lr1e4_instr': dict(model=f'{DRIVE_ROOT}/runs/scale_two_instr8k_lr1e4/20261027-144400-b2b0c4b/final', "
      "query_prompt=STUDENT_INSTR, reuse_from='d_two_instr8k_lr1e4'),   # control")


In [ ]:
from google.colab import runtime
runtime.unassign()